In [1]:
from KG.kg import ingest_Chunks, create_nodes, create_relationship, create_vector_index, embed_text
from KG.chunking import split_data_from_file
from KG.config import load_neo4j_graph
import json
graph, gemini_api, _ = load_neo4j_graph()

In [2]:
file_names = ["Talleyrand", "Napoleon", "Battle_of_Waterloo"]

for name in file_names:
    #  Load JSON file
    file = f"data/{name}.json"
    # Chunking
    chunks = split_data_from_file(file)
    # Assuming `file` is a path to your JSON file
    with open(file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    if name == "Battle_of_Waterloo":
        create_nodes(graph=graph, data=data, node_label="Event", node_name=name)
    else:
        create_nodes(graph=graph, data=data, node_label="Person", node_name=name)
    # Ingest Chunks
    ingest_Chunks(graph=graph, chunks=chunks, node_name=name, node_label='Chunk')

['General Information', 'Career', 'Death', 'Source']
Processing General Information from data/Talleyrand.json
	Split into 12 chunks
Processing Career from data/Talleyrand.json
	Split into 13 chunks
Processing Death from data/Talleyrand.json
	Split into 1 chunks
Processing Source from data/Talleyrand.json
	Split into 1 chunks
Creating `:Chunk` node for chunk ID Talleyrand-General Information-chunk0000
Creating `:Chunk` node for chunk ID Talleyrand-General Information-chunk0001
Creating `:Chunk` node for chunk ID Talleyrand-General Information-chunk0002
Creating `:Chunk` node for chunk ID Talleyrand-General Information-chunk0003
Creating `:Chunk` node for chunk ID Talleyrand-General Information-chunk0004
Creating `:Chunk` node for chunk ID Talleyrand-General Information-chunk0005
Creating `:Chunk` node for chunk ID Talleyrand-General Information-chunk0006
Creating `:Chunk` node for chunk ID Talleyrand-General Information-chunk0007
Creating `:Chunk` node for chunk ID Talleyrand-General In

In [3]:
# Create relationship
rel_section_chunk = """ 
MATCH (s:Section), (c:Chunk)
WHERE s.type = c.source AND s.parent_name = c.node_name
MERGE (s)-[:HAS_CHUNK]->(c);

"""

rel_person_person = """
MATCH (p1:Person), (p2:Person)
WHERE id(p1) < id(p2)
MERGE (p1)-[:RELATED_TO]->(p2)
MERGE (p2)-[:RELATED_TO]->(p1);

"""

rel_person_event = """
MATCH (p:Person), (e:Event)
MERGE (p)-[:RELATED_TO]->(e)
MERGE (e)-[:RELATED_TO]->(p);

"""

rel_person_section = """
MATCH (p:Person), (s:Section)
WHERE p.name = s.parent_name
MERGE (p)-[:HAS_SECTION]->(s);

"""

rel_event_section = """
MATCH (e:Event), (s:Section)
WHERE e.name = s.parent_name
MERGE (e)-[:HAS_SECTION]->(s);

"""

queries = [rel_section_chunk, rel_person_person, rel_person_event, rel_person_section, rel_event_section]

for query in queries:
    create_relationship(graph=graph, query=query)

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. id is deprecated. It is replaced by elementId or consider using an application-generated id.', position=<SummaryInputPosition line=3, column=7, offset=38>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 38, 'line': 3, 'column': 7}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (p1:Person), (p2:Person)\nWHERE id(p1) < id(p2)\nMERGE (p1)-[:RELATED_TO]->(p2)\nMERGE (p2)-[:RELATED_TO]->(p1);\n\n'
Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. id is deprecated. It is replaced by elementId or c

In [4]:
create_vector_index(graph=graph, index_name='Chunk')

In [6]:
embed_text(graph=graph, api_key=gemini_api, node_name='Chunk', batch_size=32)

Starting embedding update...


[#DD64]  _: <CONNECTION> error: Failed to read from defunct connection ResolvedIPv4Address(('34.126.161.242', 7687)) (ResolvedIPv4Address(('34.126.161.242', 7687))): OSError('No data')
Unable to retrieve routing information
Transaction failed and will be retried in 0.9677926538755972s (Unable to retrieve routing information)
[#F18F]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('p-mt-2a41353b6b9a-16-0108.production-orch-0695.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.161.242', 7687))): OSError('No data')
Transaction failed and will be retried in 2.390264697519908s (Failed to read from defunct connection IPv4Address(('p-mt-2a41353b6b9a-16-0108.production-orch-0695.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.161.242', 7687))))


Found 22 nodes without embeddings.


Embedding: 100%|██████████████████████████████████████████████████████| 1/1 [00:08<00:00,  8.68s/it]

Finished embedding update.
